# 02. Retriever 테스트

**목적**: `app/retriever.py`의 벡터스토어 구축 및 검색 동작 확인

**체크리스트**
- [ ] 문서 → 청크 분할 확인
- [ ] 벡터스토어 구축 확인
- [ ] 유사도 검색 결과 확인
- [ ] 유사도 점수 임계값 동작 확인

In [ ]:
import sys
sys.path.insert(0, '..')

## 1. 벡터스토어 구축

In [ ]:
from app.retriever import build_vectorstore, get_retriever

documents = [
    "인공지능(AI)은 컴퓨터 시스템이 인간의 지능을 모방하여 학습하고 추론할 수 있도록 하는 기술입니다.",
    "머신러닝은 AI의 한 분야로, 데이터로부터 패턴을 학습하여 예측이나 분류를 수행합니다.",
    "딥러닝은 신경망을 사용하여 복잡한 패턴을 학습하는 머신러닝의 하위 분야입니다.",
    "자연어처리(NLP)는 인간의 언어를 컴퓨터가 이해하고 처리할 수 있도록 하는 AI 기술입니다.",
    "컴퓨터 비전은 이미지나 비디오에서 의미 있는 정보를 추출하는 AI 분야입니다.",
]

vectorstore = build_vectorstore(documents)
print(f"벡터스토어 구축 완료")

## 2. 유사도 검색

In [ ]:
query = "컴퓨터 비전에 대해 알려줘"
docs_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print(f"쿼리: {query}\n")
for i, (doc, score) in enumerate(docs_with_scores):
    print(f"[{i+1}] 점수: {score:.4f}")
    print(f"     내용: {doc.page_content}\n")

## 3. 임계값 필터링 확인

In [ ]:
from app.config import settings

off_topic_query = "오늘 날씨가 좋아요"
docs_with_scores = vectorstore.similarity_search_with_score(off_topic_query, k=3)

print(f"쿼리: {off_topic_query}")
print(f"유사도 임계값: {settings.SIMILARITY_THRESHOLD}\n")
for i, (doc, score) in enumerate(docs_with_scores):
    status = "통과" if score >= settings.SIMILARITY_THRESHOLD else "필터링"
    print(f"[{i+1}] 점수: {score:.4f} → {status}")
    print(f"     내용: {doc.page_content[:60]}...\n")

## 4. Retriever 객체 테스트

In [ ]:
retriever = get_retriever(vectorstore, k=2)
results = retriever.invoke("신경망 학습")

print(f"검색 결과 {len(results)}건:")
for doc in results:
    print(f" - {doc.page_content}")